We started by copy/pasting the literal whole assignment 3 to AI with the following prompt:  
"_Do not program anything right now, but brainstorm how to approach this assignment_"

And we got this answer:  
>This is a great assignment—you're essentially building a "Baby's First Language Model" from scratch. It takes you through the entire lifecycle, from raw, messy data all the way to a trained, evaluated neural network, wrapped up nicely in a Jupyter Notebook.  
Here is our overarching roadmap for tackling all nine steps:  
Step 1: The Great Extraction. Pulling the raw text out of the PDF.  
Step 2: The Lego Phase (Tokenization). Using tiktoken to chop the massive wall of text into digestible, mathematical pieces.  
Step 3: Creating the Sliding Window. Structuring those tokens into sequences of 100 so the model has a context window to learn from.  
Step 4: The 80/20 Split. Dividing our data pile into study material (training) and a pop quiz (validation/testing).  
Step 5: Architecting the Brain & The Theory. Setting up the neural network and explaining the critical difference between a dense layer and an embedding layer.  
Step 6: The Report Card. Tracking the training/validation loss and accuracy to make sure the model is actually learning, not just memorizing.  
Step 7: The Litmus Test. Letting the trained model generate text to see what it learned.  
Step 8: Keeping the Receipts. Commenting the code and documenting the AI prompts used to generate the snippets.  
Step 9: Packaging. Formatting everything cleanly into a Jupyter Notebook with Markdown blocks explaining the "why" behind the code.  
Let's brainstorm this first hurdle.  
PDFs are notoriously stubborn. They aren't structured text documents; they are essentially digital photographs of a page. A PDF knows that the letter "A" is at coordinate (X, Y), but it doesn't inherently understand paragraphs, columns, or page breaks.  
If we use a basic tool, we might accidentally pull in page numbers, headers, footers, or weird line breaks right in the middle of sentences, which will seriously confuse your neural network later on.  
To tackle this, we need to think about which Python library will do the heavy lifting. PyPDF2 is the classic choice, but it's famously clumsy. PyMuPDF (also known as fitz) or pdfplumber are generally much better at reading the text as a human would, going left-to-right and top-to-bottom.

From this we started looking into pdf extractors as the formatting hurdle was not even something we had thought of. 

We decided to go with `pymupdf4llm`.

In [ ]:
## STEP 1: Code block that reads this course book in pdf to text ##

import pymupdf4llm

# TAKING THE BOOK FROM OUR PROJECT FOLDER
pdf_path = "book.pdf"

# EXTRACTING THE TEXT FROM THE PDF
book_text = pymupdf4llm.to_markdown(pdf_path)

# CHECKING THE FIRST 500 CHARACTERS OF THE EXTRACTED TEXT IF IT SUCCESSFULLY EXTRACTED
print("----- EXTRACTION SUCCESSFUL -----\n 500 first characters:\n")
print(book_text[:500])

=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

Answer after **1 min 5.3 seconds** :
```
Extracting text...

----- EXTRACTION SUCCESSFUL -----
 500 first characters:

## SECOND EDITION 

## François Chollet 

**==> picture [43 x 34] intentionally omitted <==**

**M A N N I N G** 

_Deep Learning with Python_ 

**Licensed to kari Jyrkkä <kari.jyrkka@oamk.fi>** 

**Licensed to kari Jyrkkä <kari.jyrkka@oamk.fi>** 

_Deep Learning with Python_ 

**SECOND EDITION** FRANÇOIS CHOLLET 

M A N N I N G SHELTER ISLAND 

**Licensed to kari Jyrkkä <kari.jyrkka@oamk.fi>** 

For online information and ordering of this and other Manning books, please visit www.manning.com. T
```

Asked AI what extention method would be best for our project and it said to use Markdown, we asked why Markdown with the following prompt:

"_Is there a reason we are extracting for markdown instead of just plain text?_"

Answer:  
>Yes, there is a massive, strategic reason we are using Markdown instead of standard plain text.  
The CVS Receipt vs. The Blueprint  
Extracting a 500-page technical textbook as raw, plain text is essentially turning it into a giant, unformatted CVS receipt. All the structural hierarchy—chapter titles, subheadings, bullet points, and code blocks—gets violently flattened into one endless, monotonous string of characters.
Markdown, on the other hand, preserves the digital blueprint.  
Spatial Awareness: It keeps headers as # Headers. When we eventually feed this data to your neural network in chunks, those hashtags act as little road signs. It helps the AI learn the underlying structure of a book, not just the vocabulary.  
The Code Factor (The Real MVP): This is the biggest win for your specific book. Chollet’s book is packed with Python scripts. Plain text will mash his English paragraphs right into his Python functions. Markdown, however, wraps the code beautifully in ```python tags. Your neural network will literally learn to distinguish between conversational English and Python syntax much faster because we preserved those structural boundaries.

Asked AI what encoder to use, and it suggested cl100k_base so we asked why that specifically, and got answer: 

>Great question. In the world of Large Language Models, cl100k_base is essentially the modern gold standard dictionary. Specifically, it is the exact tokenizer encoding that OpenAI uses for GPT-4 and modern ChatGPT.  
Here is the real-life breakdown of why we chose this specific one instead of an older one:  
The Efficiency Upgrade: Think of older tokenizers (like r50k_base used for GPT-2) as a 1990s dictionary. If you try to explain the concept of a "smartphone" to someone in 1990, it takes a whole paragraph of words (and therefore, a ton of tokens) to describe it. cl100k_base is the modern dictionary—it has updated its vocabulary to include more modern words and common word-groupings as single, highly efficient tokens.  
The Code Cruncher: Since you are processing François Chollet's Python-heavy textbook, this is crucial. cl100k_base is highly optimized to read coding languages. It groups common Python syntax (like sequential spaces, tabs, and keywords like import or def) into single, clean numbers instead of shattering them into dozens of tiny, useless pieces.

In [ ]:
## STEP 2: Use tiktokenizer to make the text to tokens ##

import tiktoken

# CHOOSING THE ENCODER FOR OUR MODEL (CL100K_BASE USED BY GPT-4 AND GPT-3.5)
encoder = tiktoken.get_encoding("cl100k_base")

# TOKENIZING THE BOOK TEXT
tokens = encoder.encode(book_text)

# CHECKING THE TOKENIZATION RESULTS
print("----- TOKENIZATION SUCCESSFUL -----")
print(f"Total tokens in the book: {len(tokens):,}")
print(f"First 10 tokens (the actual numbers): {tokens[:10]}")

# DECODE THE FIRST 10 TOKENS TO SEE WHAT THEY REPRESENT IN TEXT
print(f"What those first 10 numbers mean: {repr(encoder.decode(tokens[:10]))}")

----- TOKENIZATION SUCCESSFUL -----
Total tokens in the book: 270,900
First 10 tokens (the actual numbers): [567, 69795, 469, 10193, 4815, 567, 85807, 921, 337, 1169]
What those first 10 numbers mean: '## SECOND EDITION \n\n## François Chollet'


Answer after **0.2 seconds**:

```
----- TOKENIZATION SUCCESSFUL -----
Total tokens in the book: 270,900
First 10 tokens (the actual numbers): [567, 69795, 469, 10193, 4815, 567, 85807, 921, 337, 1169]
What those first 10 numbers mean: '## SECOND EDITION \n\n## François Chollet'
```

In [24]:
## STEP 3: Create training data. Use context length = 100 tokens ##

import numpy as np

# SETTING THE CONTEXT LENGTH TO 100 TOKENS
context_length = 100

# CREATING THE TRAINING DATA: X will be the context (100 tokens) and y will be the next token (the answer key)
X = []
y = []

# LOOP THROUGH THE TOKENS TO CREATE 100 TOKEN CONTEXT WINDOWS AND THE NEXT TOKEN AS THE TARGET
for i in range(0, len(tokens) - context_length):
    # GRAB A CONTEXT WINDOW OF 100 TOKENS
    context_window = tokens[i : i + context_length]
    # GRAB THE NEXT TOKEN AS THE TARGET
    target_token = tokens[i + context_length]
    
    X.append(context_window)
    y.append(target_token)

# CONVERTING THE LISTS TO NUMPY ARRAYS FOR EASIER HANDLING IN THE MODEL
X = np.array(X)
y = np.array(y)

print("----- TRAINING DATA CREATED -----")
print(f"Shape of X (Inputs): {X.shape} -> (Number of training examples, Context length)")
print(f"Shape of y (Targets): {y.shape} -> (Number of answer keys,)")

----- TRAINING DATA CREATED -----
Shape of X (Inputs): (270800, 100) -> (Number of training examples, Context length)
Shape of y (Targets): (270800,) -> (Number of answer keys,)


Answer after **0.9 seconds**: 

```
----- TRAINING DATA CREATED -----
Shape of X (Inputs): (270800, 100) -> (Number of training examples, Context length)
Shape of y (Targets): (270800,) -> (Number of answer keys,)
```

In [25]:
## STEP 4: Split your training data to test and validation with 80/20 split ##

# CALCULATE THE INDEX TO SPLIT THE DATA (80% FOR TRAINING, 20% FOR VALIDATION)
split_index = int(0.8 * len(X))

# SLICE THE DATA INTO TRAINING AND VALIDATION SETS
X_train = X[:split_index]
y_train = y[:split_index]

# VALIDATION DATA
X_val = X[split_index:]
y_val = y[split_index:]

print("----- DATA SPLIT SUCCESSFUL -----")
print(f"Training Data (80%): {len(X_train):,}")
print(f"Validation Data (20%): {len(X_val):,}")

----- DATA SPLIT SUCCESSFUL -----
Training Data (80%): 216,640
Validation Data (20%): 54,160


Answer after **0.0 seconds**:
```
----- DATA SPLIT SUCCESSFUL -----
Training Data (80%): 216,640
Validation Data (20%): 54,160
```

In [ ]:
## STEP 5a: Define simple neural network and teach it with your data. ##

import tensorflow as tf
from tensorflow.keras import layers, models

# CHECKING THE VOCABULARY SIZE OF OUR ENCODER TO KNOW HOW MANY UNIQUE TOKENS THERE ARE
vocab_size = encoder.n_vocab

# BUILDING THE SIMPLE NEURAL NETWORK MODEL
model = models.Sequential([
    # FIRST LAYER: EMBEDDING LAYER TO TURN TOKEN NUMBERS INTO DENSE VECTORS
    layers.Embedding(input_dim=vocab_size, output_dim=32, input_length=context_length),
    
    # SECOND LAYER: LSTM LAYER TO CAPTURE SEQUENTIAL PATTERNS IN THE TOKEN CONTEXT
    layers.LSTM(32), # CAN BE CHANGED TO 128 OR MORE FOR TESTING 

    # OUTPUT LAYER: DENSE LAYER WITH SOFTMAX ACTIVATION TO PREDICT THE PROBABILITY OF EACH TOKEN IN THE VOCABULARY BEING THE NEXT TOKEN
    layers.Dense(vocab_size, activation='softmax')
])

# COMPILING THE MODEL WITH AN OPTIMIZER, LOSS FUNCTION AND METRICS TO TRACK
model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', # THIS LOSS FUNCTION IS USED FOR MULTI-CLASS CLASSIFICATION PROBLEMS WHERE THE TARGETS ARE INTEGER ENCODED
    metrics=['accuracy']
)

# SHOWING THE MODEL SUMMARY TO SEE THE ARCHITECTURE AND NUMBER OF PARAMETERS
model.build(input_shape=(None, 100))
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 100, 32)        │     3,208,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 100277)         │    12,935,733 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,227,029 (61.90 MB)

 Trainable params: 16,227,029 (61.90 MB)

 Non-trainable params: 0 (0.00 B)

Answer after **1.8 seconds**:

```
Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 100, 32)        │     3,208,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100277)         │     3,309,141 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

Total params: 6,526,325 (24.90 MB)

Trainable params: 6,526,325 (24.90 MB)

Non-trainable params: 0 (0.00 B)
```

In [ ]:
# MOVING THIS TO ITS OWN CELL SO IT CAN BE RUN MULTIPLE TIMES WITHOUT REDEFINING THE MODEL
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=1,
    batch_size=128 # USING A BATCH SIZE OF 128 TO SPEED UP TRAINING.
)

Epoch 1/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 222s 131ms/step - accuracy: 0.4209 - loss: 2.9268 - val_accuracy: 0.2597 - val_loss: 6.2247
Epoch 2/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 220s 130ms/step - accuracy: 0.4320 - loss: 2.8359 - val_accuracy: 0.2575 - val_loss: 6.3284
Epoch 3/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 220s 130ms/step - accuracy: 0.4426 - loss: 2.7552 - val_accuracy: 0.2593 - val_loss: 6.4072
Epoch 4/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 224s 132ms/step - accuracy: 0.4543 - loss: 2.6789 - val_accuracy: 0.2603 - val_loss: 6.5016
Epoch 5/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 222s 131ms/step - accuracy: 0.4632 - loss: 2.6117 - val_accuracy: 0.2590 - val_loss: 6.5684
Epoch 6/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 220s 130ms/step - accuracy: 0.4724 - loss: 2.5458 - val_accuracy: 0.2600 - val_loss: 6.6440
Epoch 7/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 219s 130ms/step - accuracy: 0.4815 - loss: 2.4871 - val_accuracy: 0.2593 - val_loss: 6.7283
Epoch 8/10
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 219s 130ms/step - ac

Answer after **2 min 5.1 seconds**: 
```
1693/1693 ━━━━━━━━━━━━━━━━━━━━ 121s 71ms/step - accuracy: 0.0359 - loss: 7.1823 - val_accuracy: 0.0442 - val_loss: 7.3041
```

### STEP 5b: Explain why simple dense layer is not a good layer in LLM.
If we feed raw token IDs (like 567 for "Apple" and 568 for "Car") into a standard Dense layer the neural network will treat them as pure math and incorrectly assume that "Car" is mathematically greater than "Apple" because 568 numerically is bigger than 567.  
To fix that, older models used "One-Hot Encoding" creating a list of 100,000 zeros with a single 1 for the correct word. If 100 words are passed into a Dense layer a lot of useless zeros get multiplied all the time which slows down the process and might crash the whole system.
Dense layers also have no concept of "meaning" in words. Like "King" and "Queen" are related concepts.

### STEP 5c: What is `layers.Embedding` that chatGPT will propose for you?
Instead of treating tokens as arbitrary numbers or massive arrays of zeros, it assigns every single token a specific mathematical coordinate like a vector in a dense multidimensional space.  
As the model trains, it physically moves the coordinates of words that are used in similar contexts closer together. "Dog" and "Puppy" end up in the same corner of the math space. It allows the model to understand context, nuance, and relationships, while being incredibly memory-efficient.

In [46]:
## STEP 6: Print train loss, val_loss, accuracy and validation accuracy ##

# EXTRACTING THE FINAL NUMBERS FROM OUR TRAINING HISTORY
# CURRENTLY ONLY HAVE 1 EPOCH, SO WE TAKE INDEX [0] TO GET THE FINAL NUMBERS. 
# IF THERE WOULD HAVE BEEN MORE EPOCHS TAKE THE LAST EPOCH TO SEE THE FINAL SCORE
train_loss = history.history['loss'][0]
val_loss = history.history['val_loss'][0]
train_acc = history.history['accuracy'][0]
val_acc = history.history['val_accuracy'][0]

# PRINTING THE GOODIES
print("----- PRINTED TRAINING RESULTS -----")
print(f"Training Loss:       {train_loss:.4f}")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Training Accuracy:   {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

----- PRINTED TRAINING RESULTS -----
Training Loss:       2.9268
Validation Loss:     6.2247
Training Accuracy:   0.4209
Validation Accuracy: 0.2597


Answer after **0.0 seconds**:
```
----- PRINTED TRAINING RESULTS -----
Training Loss:       7.1823
Validation Loss:     7.3041
Training Accuracy:   0.0359
Validation Accuracy: 0.0442
```

In [ ]:
## STEP 7: Test you model after training ##

from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# BUILDING A FUNCTION TO GENERATE TEXT FROM THE MODEL GIVEN A SEED PROMPT
def generate_text(seed_text, num_words=30):
    # TRANSLATING OUR STARTING ENGLISH PROMPT INTO INTEGER TOKENS
    input_tokens = encoder.encode(seed_text)
        
    for _ in range(num_words):
        # MODEL STRICTLY EXPECTS EXACTLY 100 TOKENS. 
        # IF OUR PROMPT IS TOO SHORT THE FRONT WILL BE FILLED WITH ZEROS.
        padded_input = pad_sequences([input_tokens], maxlen=context_length, padding='pre', truncating='pre')
        
        # ASKING THE MODEL FOR THE PROBABILITIES OF ALL 100,000+ POSSIBLE NEXT WORDS
        predicted_probs = model.predict(padded_input, verbose=0)
        
        # PICKING THE ABSOLUTE HIGHEST PERCENTAGE GUESS (ARGMAX)
        best_guess_token = np.argmax(predicted_probs, axis=-1)[0]
        
        # ADDING THAT NEW TOKEN TO THE RUNNING LIST SO IT CAN BE USED FOR THE NEXT GUESS
        input_tokens.append(best_guess_token)
        
    # TRANSLATING THE FINAL LIST OF MATH BACK INTO HUMAN TEXT
    return encoder.decode(input_tokens)

# TESTING THE MODEL WITH A SEED PROMPT TO SEE WHAT IT CAN GENERATE
print("----- MODEL GENERATION -----")
my_prompt1 = "Deep learning is "
generated_output1 = generate_text(my_prompt1, num_words=30)
my_prompt2 = "One, two, three, "
generated_output2 = generate_text(my_prompt2, num_words=30)

print("Final Output:")
print("--------------------------------------------------")
print(generated_output1)
print("--------------------------------------------------")
print(generated_output2)
print("--------------------------------------------------")


----- MODEL GENERATION -----
Final Output:
--------------------------------------------------
Deep learning is 2013, although you’ll be successfully optimized, but although deep learning models, you’re already familiar with the field of deep learning. 

## _
--------------------------------------------------
One, two, three, 2013, 2013, 2015, 2015, 2015, 2015, 2015, 2016
--------------------------------------------------


Answer after **2.5 seconds**:
```
----- MODEL GENERATION -----
Final Output:
--------------------------------------------------
Deep learning is  the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
--------------------------------------------------
One, two, three,  the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
--------------------------------------------------
```

### Note on the output

The model generating a repetitive loop of "_the_" is the mathematically expected behavior here.  
In Step 5 the model was only trained for a single epoch (only about 10 min) using a very lightweight LSTM architecture. The high training loss (7.17) and low accuracy (3.7%) from Step 6 confirmed to us that the network did not learn the underlying grammar or context of the tokens making it fall back on predicting the most statistically frequent token in English which apparently in this case is "_the_"

In [ ]:
## CHANGE delete this block when done
train_loss = history.history['loss'][-1]
val_loss = history.history['val_loss'][-1]
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

# PRINTING THE GOODIES
print("----- PRINTED TRAINING RESULTS -----")
print(f"Training Loss:       {train_loss:.4f}")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Training Accuracy:   {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

----- PRINTED TRAINING RESULTS -----
Training Loss:       2.3258
Validation Loss:     6.9635
Training Accuracy:   0.5065
Validation Accuracy: 0.2564


# Running more training

### 1 + 10 epochs & `layers.LSTM(32)`
```
----- PRINTED TRAINING RESULTS -----
Training Loss:       6.3956
Validation Loss:     6.9167
Training Accuracy:   0.0782
Validation Accuracy: 0.0818


----- MODEL GENERATION -----
Final Output:
--------------------------------------------------
Deep learning is 2011, 2011, 2011, 2011, 2011, 2011, 2011, 2011
--------------------------------------------------
One, two, three, 2011, 2011, 2011, 2011, 2011, 2011, 2011, 2011
--------------------------------------------------
```

### 11 + 10 epochs & `layers.LSTM(32)`
```
----- PRINTED TRAINING RESULTS -----
Training Loss:       4.0543
Validation Loss:     6.0397
Training Accuracy:   0.3075
Validation Accuracy: 0.2192


----- MODEL GENERATION -----
Final Output:
--------------------------------------------------
Deep learning is 1.1.1.1. 

- _The ___ ,000 data, you can be able to the model’s a single way
--------------------------------------------------
One, two, three, 2012, 0] 

- _The 2D._ ■ _The ___ ,000 of the model. 

**Licensed
--------------------------------------------------
```

### 1 epoch & `layers.LSTM(128)`
```
----- PRINTED TRAINING RESULTS -----
Training Loss:       6.8445
Validation Loss:     6.7595
Training Accuracy:   0.0613
Validation Accuracy: 0.1083


----- MODEL GENERATION -----
Final Output:
--------------------------------------------------
Deep learning is  

 

 

 

 

** 

**********************************************
--------------------------------------------------
One, two, three,  

 

 

 

 

** 

**********************************************
--------------------------------------------------
```

### 1 + 10 epochs & `layers.LSTM(128)`
```
----- PRINTED TRAINING RESULTS -----
Training Loss:       3.0260
Validation Loss:     6.1508
Training Accuracy:   0.4078
Validation Accuracy: 0.2580


----- MODEL GENERATION -----
Final Output:
--------------------------------------------------
Deep learning is 3_ ■ ______ _The machine learning workspace. 

- _The promise of machine learning_ 

- _The promise of
--------------------------------------------------
One, two, three, 2017, a good model that you’re trying to be able to train the model on the training data. 

- _Metrics_ —The first
--------------------------------------------------
```

### 11 + 10 epochs & `layers.LSTM(128)`
```
----- PRINTED TRAINING RESULTS -----
Training Loss:       2.3258
Validation Loss:     6.9635
Training Accuracy:   0.5065
Validation Accuracy: 0.2564


----- MODEL GENERATION -----
Final Output:
--------------------------------------------------
Deep learning is 2013, although you’ll be successfully optimized, but although deep learning models, you’re already familiar with the field of deep learning. 

## _
--------------------------------------------------
One, two, three, 2013, 2013, 2015, 2015, 2015, 2015, 2015, 2016
--------------------------------------------------
```